# CALOR Y MAYORES EN MADRID

## Limpieza de datos: centros de mayores

**Fuente:** Registro de centros de atención social, Comunidad de Madrid (CSV, versión posterior a septiembre de 2026).

**Objetivo:** quedarnos con las residencias, los centros de día y los centros de mayores de Madrid capital, con coordenadas, plazas y titularidad revisadas.

## 1. Carga y revisión general

In [17]:
import pandas as pd
import numpy as np
import geopandas as gpd

In [2]:
centers_raw = pd.read_csv("../raw/dato2y3/atencion_social_registro_centros.csv", sep=";", encoding="utf-8")
centers = centers_raw.copy()
centers.head()

,numero_registro,sector,tipo,subtipo,denominacion,plazas_autorizadas_numero,fecha_acreditacion,fecha_renovacion_acreditacion,municipio_codigo,municipio_nombre,...,direccion_demarcacion,direccion_zonass,direccion_coor_x,direccion_coor_y,contacto_telefono,contacto_correo,entidad_denominacion,entidad_nif,entidad_naturaleza,tipo_gestion
0,C0003,Personas con enfermedad mental crónica,CENTROS DE REHABILITACIÓN,CENTROS DE REHABILITACIÓN: OTROS,CENTRO DE REHABILITACION LABORAL NUEVA VIDA,40.0,NaN,NaN,079,Madrid,...,Salamanca 1,Guindalera,443644,4476188,91355 26 80,crl@crlnuevavida.es;pcarreras@crlnuevavida.es,ASOCIACION PSIQUIATRIA Y VIDA,G28727261,PRIVADA SIN ÁNIMO DE LUCRO,PRIVADA CON PLAZAS CONCERTADAS
1,C0005,Personas mayores,CENTROS RESIDENCIALES,RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES,"NTRA. SRA. DE FATIMA, RESIDENCIA DE PERSONAS M...",96.0,27-05-2015,17-11-2026,061,Galapagar,...,San Lorenzo de El Escorial,Galapagar,415078,4491976,91 858 42 40,fatima@residencia-fatima.com;marina.hermoso@as...,"RESIDENCIA FATIMA, S.L.",B79025193,PRIVADA CON ÁNIMO DE LUCRO,PRIVADA + PEVS
2,C0007,Personas mayores,CENTROS RESIDENCIALES,RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES,"SAN CAMILO, RESIDENCIA DE PERSONAS MAYORES",144.0,20-08-2026,20-08-2031,903,Tres Cantos,...,Tres Cantos,Tres Cantos I,439602,4496392,91 803 40 00,personas@sancamilo.org;residencia@sancamilo.or...,RR. CAMILOS-RESIDENCIA ASISTIDA SAN CAMILO,R2800714D,PRIVADA SIN ÁNIMO DE LUCRO,PRIVADA CON PLAZAS CONCERTADAS + PEVS
3,C0010,Personas mayores,CENTROS DE DÍA,CENTROS DE DÍA: OTROS,"NUEVA EDAD, CENTRO DE DIA",60.0,20-05-2025,20-05-2030,079,Madrid,...,Ciudad Lineal 3,San Pascual,444737,4476780,914 033 131/902 095 902,asispa@asispa.org;vheranz@asispa.org,ASOCIACION SERVICIO INTEGRAL SECTORIAL PARA AN...,G28702504,PRIVADA SIN ÁNIMO DE LUCRO,PRIVADA CON PLAZAS CONCERTADAS + PEVS
4,C0018,Personas mayores,CENTROS RESIDENCIALES,RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES,"MADRE MARAVILLAS, RESIDENCIA DE PERSONAS MAYORES",54.0,20-08-2025,20-08-2030,065,Getafe,...,Zona Sur Rural,Perales del Río,444260,4463944,916847085-659381608,info@madremaravillas.com,OPTIMA CONSULTING 2024 S.L,B19352210,PRIVADA CON ÁNIMO DE LUCRO,PRIVADA CON PLAZAS CONCERTADAS + PEVS


In [8]:
centers.info()
print(centers.shape)

<class 'pandas.DataFrame'>
RangeIndex: 2636 entries, 0 to 2635
Data columns (total 28 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   numero_registro                2636 non-null   str    
 1   sector                         2636 non-null   str    
 2   tipo                           2636 non-null   str    
 3   subtipo                        2634 non-null   str    
 4   denominacion                   2636 non-null   str    
 5   plazas_autorizadas_numero      2632 non-null   float64
 6   fecha_acreditacion             1370 non-null   str    
 7   fecha_renovacion_acreditacion  1370 non-null   str    
 8   municipio_codigo               2636 non-null   str    
 9   municipio_nombre               2636 non-null   str    
 10  direccion_via_tipo             2636 non-null   str    
 11  direccion_via_codigo           2630 non-null   str    
 12  direccion_via_nombre           2636 non-null   str    
 13 

In [4]:
(centers.isnull().sum() / len(centers) * 100).round(2).sort_values(ascending=False)

direccion_mancomunidad           86.08
fecha_acreditacion               48.03
fecha_renovacion_acreditacion    48.03
contacto_correo                   5.35
direccion_coor_y                  3.22
direccion_coor_x                  3.22
entidad_nif                       0.76
contacto_telefono                 0.68
direccion_via_codigo              0.23
plazas_autorizadas_numero         0.15
subtipo                           0.08
direccion_demarcacion             0.08
direccion_zonass                  0.08
direccion_distrito                0.04
direccion_area                    0.04
municipio_codigo                  0.00
sector                            0.00
numero_registro                   0.00
tipo                              0.00
denominacion                      0.00
direccion_via_tipo                0.00
municipio_nombre                  0.00
direccion_codigo_postal           0.00
direccion_via_nombre              0.00
direccion_portal                  0.00
entidad_denominacion     

**Observación:** solo faltan datos relevantes en las coordenadas (3,2 %); direccion_mancomunidad (86 %) y las fechas de acreditación (48 %) no se usan en el proyecto.

In [5]:
centers["sector"].value_counts(dropna=False)

sector
Personas mayores                                        1072
Personas con discapacidad intelectual                    382
Personas con enfermedad mental crónica                   226
Otros grupos en situación de necesidad                   156
Personas sin hogar                                       153
Refugiados y asilados                                    137
Infancia y Adolescencia                                  134
Mujer                                                     77
Familia                                                   70
Infancia                                                  66
Adolescencia                                              63
Personas con discapacidad física                          57
(Inmigracion) Otros grupos en situación de necesidad      26
Presos y ex reclusos                                      12
Conjunto de la población                                   4
Colectivos en situación de necesidad                       1
Name: count, dtyp

**Observación:** 1.072 centros son de personas mayores, el sector que nos interesa; el resto (discapacidad, infancia, sin hogar...) queda fuera del estudio.

In [6]:
centers["tipo"].value_counts(dropna=False)

tipo
CENTROS RESIDENCIALES                       1565
CENTROS DE DÍA                               724
CENTROS DE REHABILITACIÓN                    205
CENTROS OCUPACIONALES                         93
CENTROS DE ACOGIDA                            48
CENTROS EXPERIMENTALES O NO CLASIFICADOS       1
Name: count, dtype: int64

**Observación:** nos quedaremos con centros residenciales y centros de día, que incluyen las residencias, los centros de día y los hogares y clubes.

In [7]:
centers["municipio_nombre"].value_counts(dropna=False).head(10)

municipio_nombre
Madrid                        1275
RESERVADO                      172
Leganés                         88
Móstoles                        57
Getafe                          56
Alcalá de Henares               56
Pozuelo de Alarcón              50
San Sebastián de los Reyes      45
Fuenlabrada                     41
Alcobendas                      38
Name: count, dtype: int64

**Observación:** Madrid capital tiene 1.275 centros; los 172 RESERVADO son centros protegidos (sobre todo de mujer e infancia) y quedan fuera al filtrar por Madrid y mayores.

## 2. Filtrado

- nos quedamos con los centros de Madrid capital, de personas mayores, residenciales o de día

In [9]:
centers = centers[
    (centers["municipio_nombre"] == "Madrid") &
    (centers["sector"] == "Personas mayores") &
    (centers["tipo"].isin(["CENTROS RESIDENCIALES", "CENTROS DE DÍA"]))
]
print(centers.shape)

(450, 28)


In [10]:
centers["subtipo"].value_counts(dropna=False)

subtipo
CENTROS DE DÍA: OTROS                           158
HOGARES Y CLUBES                                116
RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES    105
RESIDENCIAS DE PERSONAS MAYORES MIXTA            59
CENTROS RESIDENCIALES: OTROS                      5
RESIDENCIAS DE PERSONAS MAYORES AUTONOMAS         4
PISOS TUTELADOS                                   3
Name: count, dtype: int64

**Observación:**quedan 450 centros de mayores en Madrid capital; 168 residencias, 158 centros de día, 116 hogares y clubes y 8 de otros tipos residenciales.

In [11]:
centers["numero_registro"].duplicated().sum()

np.int64(0)

In [12]:
print("Sin coordenadas:", centers["direccion_coor_x"].isnull().sum())
centers[["denominacion", "direccion_coor_x", "direccion_coor_y"]].head()

Sin coordenadas: 11


,denominacion,direccion_coor_x,direccion_coor_y
3,"NUEVA EDAD, CENTRO DE DIA",444737,4476780
6,EL ENCUENTRO,444238,4475696
19,"SANTA FILIPINA, RESIDENCIA DE PERSONAS MAYORES",442688,4480150
21,"RSDCIA. SACERDOTAL SAN PEDRO, RESIDENCIA DE PE...",440154,4475831
31,"NTRA. SRA. DEL PILAR, RESIDENCIA DE PERSONAS M...",448266,4477240


**Observación:**centros sin coordenadas; las demás son enteros en metros (EPSG:25830), sin coma decimal, pero guardadas como texto.

In [13]:
centers[centers["subtipo"].isin(["CENTROS RESIDENCIALES: OTROS", "PISOS TUTELADOS"])][["denominacion", "subtipo", "plazas_autorizadas_numero"]]

,denominacion,subtipo,plazas_autorizadas_numero
562,"MARIA ORUE, PISOS TUTELADOS DE PERSONAS MAYORES",PISOS TUTELADOS,28.0
735,CENTRO RESIDENCIAL PISOS TUTELADOS DE USERA,PISOS TUTELADOS,100.0
1377,"RETIRO, CENTRO RESIDENCIAL DE APARTAMENTOS MUN...",CENTROS RESIDENCIALES: OTROS,32.0
1478,"BALTASAR SANTOS, PISOS TUTELADOS",PISOS TUTELADOS,45.0
2102,APARTAMENTOS BOUCO MADRID ARAVACA,CENTROS RESIDENCIALES: OTROS,118.0
2162,LAS ARCADIAS,CENTROS RESIDENCIALES: OTROS,246.0
2346,VILLA HARMONICES MIRASIERRA,CENTROS RESIDENCIALES: OTROS,17.0
2370,VILLA HARMONICES MIRASIERRA D,CENTROS RESIDENCIALES: OTROS,16.0


**Decisión:** se excluyen los 8 pisos tutelados y apartamentos para mayores (subtipos "PISOS TUTELADOS" y "CENTROS RESIDENCIALES: OTROS"), porque son viviendas para personas autónomas y no centros de atención.

In [15]:
excluded_subtypes = ["PISOS TUTELADOS", "CENTROS RESIDENCIALES: OTROS"]
centers = centers[~centers["subtipo"].isin(excluded_subtypes)]
print(centers.shape)

(442, 28)


In [16]:
centers[centers["direccion_coor_x"].isnull()][["numero_registro", "denominacion", "subtipo", "plazas_autorizadas_numero"]]

,numero_registro,denominacion,subtipo,plazas_autorizadas_numero
119,C0456,"PADRE ELADIO, RESIDENCIA DE PERSONAS MAYORES",RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES,77.0
574,C2296,"BALLESOL MIRASIERRA, RESIDENCIA DE PERSONAS MA...",RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES,232.0
630,C2475,CENTRO DE DIA FLORENCIA,CENTROS DE DÍA: OTROS,45.0
710,C2689,BALLESOL CENTRO RESIDENCIAL PASILLO VERDE,RESIDENCIAS DE PERSONAS MAYORES MIXTA,278.0
1100,C3799,"ARTURO PAJUELO, CENTRO MUNICIPAL DE MAYORES",HOGARES Y CLUBES,210.0
1141,C3968,"PABLO CASALS, CENTRO MUNICIPAL DE DIA",CENTROS DE DÍA: OTROS,125.0
1142,C3969,"PABLO CASALS 2, CENTRO MUNICIPAL DE MAYORES",HOGARES Y CLUBES,152.0
1321,C4793,"NUESTRA SEÑORA DE LOS ANGELES, CENTRO MUNICIPA...",HOGARES Y CLUBES,350.0
1401,C5145,"PARQUE DE QUEROL, CENTRO DE DIA",CENTROS DE DÍA: OTROS,65.0
1467,C5389,"MANUEL ALEXANDRE AYUNTAMIENTO DE MADRID, CENTR...",HOGARES Y CLUBES,250.0


In [18]:
residences_cm = gpd.read_file("../raw/dato2y3/residencias.geojson")
missing_residences = ["C0456", "C2296", "C2689"]
residences_cm[residences_cm["REGISTRO"].isin(missing_residences)][["REGISTRO", "Name", "X", "Y"]]

,REGISTRO,Name,X,Y
271,C0456,Padre Eladio,438342.15625,4480738.5
284,C2296,Ballesol Mirasierra,439345.87500,4482283.5
594,C2689,Ballesol Pasillo Verde (Madrid),441778.84375,4472012.5


In [19]:
centers["direccion_coor_x"] = pd.to_numeric(centers["direccion_coor_x"], errors="coerce")
centers["direccion_coor_y"] = pd.to_numeric(centers["direccion_coor_y"], errors="coerce")
print("Sin coordenadas:", centers["direccion_coor_x"].isnull().sum())

Sin coordenadas: 11


In [20]:
recovered = residences_cm[residences_cm["REGISTRO"].isin(missing_residences)]

for index, row in recovered.iterrows():
    centers.loc[centers["numero_registro"] == row["REGISTRO"], "direccion_coor_x"] = row["X"]
    centers.loc[centers["numero_registro"] == row["REGISTRO"], "direccion_coor_y"] = row["Y"]

print("Sin coordenadas:", centers["direccion_coor_x"].isnull().sum())

Sin coordenadas: 8
